# One Claim, Three Service Lines

Claim `CLM-10058` has a total of **\$1,255.72**. An incorrect calculation after a JOIN
increases it to **\$3,767.16**.

**Grain** means what one row represents: one claim in `claims_ledger`, or one service
line in `claim_line_items`.

Run all cells from the `notebooks` folder using Python 3 and pandas.

## 1. Load the CSV files

Load both files into SQLite in memory. The source files stay unchanged.

In [1]:
import pandas as pd
import sqlite3

claims = pd.read_csv("../data/structured/raw/claims_ledger.csv")
lines = pd.read_csv("../data/structured/raw/claim_line_items.csv")

db = sqlite3.connect(":memory:")
claims.to_sql("claims_ledger", db, index=False)
lines.to_sql("claim_line_items", db, index=False);

## 2. Check the source data

In [claims_ledger.csv](../data/structured/raw/claims_ledger.csv), row **59** contains
the claim: **A59** is its ID; **I59** is its total.

In [2]:
claims.loc[claims["claim_id"] == "CLM-10058", ["claim_id", "total_billed_amount"]]

,claim_id,total_billed_amount
57,CLM-10058,1255.72


In [claim_line_items.csv](../data/structured/raw/claim_line_items.csv), rows **79–81**
contain its services. Column **B** holds the claim ID; column **F** holds each line’s
billed amount.

**\$72.48 + \$479.78 + \$703.46 = \$1,255.72.**

CSV row numbers include the header. Pandas uses separate indexes.

In [3]:
lines.loc[
    lines["claim_id"] == "CLM-10058",
    ["claim_id", "line_number", "billed_amount"]
]

,claim_id,line_number,billed_amount
77,CLM-10058,1,72.48
78,CLM-10058,2,479.78
79,CLM-10058,3,703.46


## 3. Join on claim_id

**A59** matches **B79, B80, and B81**. The JOIN returns three rows and repeats the
claim total in each. This is expected.

In [4]:
joined = pd.read_sql_query("""
SELECT c.claim_id, li.line_number, li.billed_amount,
       c.total_billed_amount
FROM claims_ledger AS c
JOIN claim_line_items AS li
    ON c.claim_id = li.claim_id
WHERE c.claim_id = 'CLM-10058'
ORDER BY li.line_number;
""", db)
joined

,claim_id,line_number,billed_amount,total_billed_amount
0,CLM-10058,1,72.48,1255.72
1,CLM-10058,2,479.78,1255.72
2,CLM-10058,3,703.46,1255.72


## 4. Incorrect SQL

`SUM(c.total_billed_amount)` adds the same claim total three times:

**\$1,255.72 × 3 = \$3,767.16.**

`GROUP BY` does not remove these repeated amounts.

In [5]:
wrong = pd.read_sql_query("""
SELECT c.claim_id,
       SUM(c.total_billed_amount) AS billed_total
FROM claims_ledger AS c
JOIN claim_line_items AS li
    ON c.claim_id = li.claim_id
WHERE c.claim_id = 'CLM-10058'
GROUP BY c.claim_id;
""", db)
wrong

,claim_id,billed_total
0,CLM-10058,3767.16


## 5. Correct SQL

Keep the original claim total. Sum `li.billed_amount` separately.

Both totals are **\$1,255.72**.

In [6]:
correct = pd.read_sql_query("""
SELECT c.claim_id,
       c.total_billed_amount AS claim_total,
       SUM(li.billed_amount) AS service_lines_total
FROM claims_ledger AS c
JOIN claim_line_items AS li
    ON c.claim_id = li.claim_id
WHERE c.claim_id = 'CLM-10058'
GROUP BY c.claim_id, c.total_billed_amount;
""", db)
correct

,claim_id,claim_total,service_lines_total
0,CLM-10058,1255.72,1255.72


## How This Affects AI

| Step | Example |
|---|---|
| Data Engineering Issue | The calculation adds a repeated claim total. |
| What AI Receives | \$3,767.16 instead of \$1,255.72. |
| Possible AI Interpretation | “The claim’s total billed amount is \$3,767.16.” |
| Business Consequence | An incorrect summary could trigger unnecessary review. |
| Architectural Control | Preserve the claim total and check it against the service-line sum. |

AI could reasonably use its input and still report the wrong amount. No AI model is run here.